# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [1]:
!pip install librosa
!pip install panns-inference

In [2]:
 !pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.0/211.0 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 46.2 MB/s eta 0:00:00


# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [3]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/345 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00002-2f1ab7b824ec75(…):   0%|          | 0.00/387M [00:00<?, ?B/s]

data/train-00001-of-00002-27425e5c1846b4(…):   0%|          | 0.00/387M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [4]:
# select the audio feature and display top three
audios = data["audio"]
audios[:3]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [5]:
import numpy as np

# select only the audio data from the dataset and store in a numpy array
audios = np.array([a["array"] for a in data["audio"]])

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [6]:
from panns_inference import AudioTagging

# load the default model into the gpu.
model = AudioTagging(checkpoint_path=None, device='mps') # change device to cpu if a gpu is not available

Checkpoint path: /root/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [7]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
# initialize connection to pinecone (get API key at app.pinecone.io)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or "YOUR_API_KEY"

In [8]:
import os
from pinecone import Pinecone

# configure client
pc = Pinecone(api_key=PINECONE_API_KEY)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [9]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [10]:
index_name = "audio-search-demo"

In [11]:
# fix colab api access key issue:

import os
from getpass import getpass

# Always re-set after runtime restart
os.environ["PINECONE_API_KEY"] = getpass("Enter PINECONE_API_KEY: ").strip()

# sanity (masked)
k = os.environ["PINECONE_API_KEY"]
print("PINECONE_API_KEY loaded:", f"{k[:6]}...{k[-4:]}" if len(k) > 12 else "(too short?)")

Enter PINECONE_API_KEY: ··········
PINECONE_API_KEY loaded: pcsk_3...Yrqw


In [12]:
# fix colab api access key issue:
from dotenv import load_dotenv, find_dotenv
import os

_ = load_dotenv(find_dotenv())

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY is not set. Run the key-entry cell above, then re-run this cell.")

print("PINECONE_API_KEY var:", f"{PINECONE_API_KEY[:6]}...{PINECONE_API_KEY[-4:]}")
print("Length:", len(PINECONE_API_KEY))

PINECONE_API_KEY var: pcsk_3...Yrqw
Length: 75


In [13]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

# quick auth check (this must work before index logic)
print("Indexes:", pc.list_indexes().names())

Indexes: ['audio-search-demo', 'abs-qa-ben-v1', 'question-answering']


In [14]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=2048,
        metric='cosine',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}

# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [15]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(audios))
    # extract batch
    batch = audios[i:i_end]
    # generate embeddings for all the audios in the batch
    _, emb = model.inference(batch)
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb.tolist()))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/32 [00:00<?, ?it/s]

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}},
 'total_vector_count': 2000}

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [ ]:
from IPython.display import Audio, display

# we set an audio number to select from the dataset
audio_num = 400
# get the audio data of the audio number
query_audio = data[audio_num]["audio"]["array"]
# get the category of the audio number
category = data[audio_num]["category"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

We have got the sound of a car horn. Let's generate an embedding for this sound.

In [17]:
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [18]:
# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)
results

{'matches': [{'id': '400', 'score': 0.999630749, 'values': []},
             {'id': '1667', 'score': 0.842086852, 'values': []},
             {'id': '1666', 'score': 0.831265926, 'values': []}],
 'namespace': '',
 'usage': {'read_units': 1}}

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results.

In [ ]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the databse using the id as an index
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [20]:
def find_similar_audios(id):
    print("Query Audio:")
    # select the audio data from the databse using the id as an index
    query_audio = data[id]["audio"]["array"]
    # play the query audio
    display(Audio(query_audio, rate=44100))
    # query pinecone index with the query audio id
    result = index.query(id=str(id), top_k=5)
    print("Result:")
    # play the top 5 similar audios
    for r in result["matches"]:
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=44100))

In [ ]:
find_similar_audios(1642)

Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [ ]:
find_similar_audios(452)

And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [23]:
!wget https://storage.googleapis.com/audioset/miaow_16k.wav

--2026-03-01 00:23:36--  https://storage.googleapis.com/audioset/miaow_16k.wav
Resolving storage.googleapis.com (storage.googleapis.com)... 192.178.210.207, 173.194.206.207, 192.178.129.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|192.178.210.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 215546 (210K) [audio/x-wav]
Saving to: ‘miaow_16k.wav’

miaow_16k.wav       100%[===================>] 210.49K  --.-KB/s    in 0.001s  

2026-03-01 00:23:36 (171 MB/s) - ‘miaow_16k.wav’ saved [215546/215546]



We can load the audio into a Numpy array as follows:

In [ ]:
import librosa

a, _ = librosa.load("miaow_16k.wav", sr=44100)
Audio(a, rate=44100)

Now we generate the embeddings for this audio and query the Pinecone index.

In [ ]:
# reshape query audio
query_audio = a[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
results = index.query(vector=xq.tolist(), top_k=3)

# play the top 3 similar audios
for r in results["matches"]:
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [ ]:
pc.delete_index(index_name)

### **Excercise Run**

In [26]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [27]:
from pathlib import Path

# Option A: you uploaded it directly into "MyDrive"
DATA_DIR = Path("/content/drive/MyDrive/1001-2000-midis")

# Option B: if you put it inside a "data" folder in Drive
# DATA_DIR = Path("/content/drive/MyDrive/data/1001-2000-midis")

print("DATA_DIR:", DATA_DIR)
print("Exists:", DATA_DIR.exists())

# show a few items to confirm we're in the right place
print("Sample files:", list(DATA_DIR.iterdir())[:5])

DATA_DIR: /content/drive/MyDrive/1001-2000-midis
Exists: True
Sample files: [PosixPath('/content/drive/MyDrive/1001-2000-midis/1873_ElectricBass.mid'), PosixPath('/content/drive/MyDrive/1001-2000-midis/1873_Drums.mid'), PosixPath('/content/drive/MyDrive/1001-2000-midis/1873_ElectricGuitarClean.mid'), PosixPath('/content/drive/MyDrive/1001-2000-midis/1873_ElectricGuitarLead.mid'), PosixPath('/content/drive/MyDrive/1001-2000-midis/1873_Flute.mid')]


In [28]:
audio_files = []
for ext in ("*.wav", "*.mp3", "*.flac", "*.ogg", "*.mid", "*.midi"):
    audio_files.extend(DATA_DIR.rglob(ext))

audio_files = sorted(audio_files)
print("Files found:", len(audio_files))
print("Example:", audio_files[0] if audio_files else "None")

Files found: 7920
Example: /content/drive/MyDrive/1001-2000-midis/1001_Balalaika.mid


In [33]:
MAX_FILES = 300  # increase later (500, 1000...) if stable
files_to_process = audio_files[:MAX_FILES]
print("Processing:", len(files_to_process))

Processing: 300


In [34]:
import shutil

LOCAL_DIR = Path("/content/data/1001-2000-midis")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

for p in files_to_process:
    dst = LOCAL_DIR / p.name
    if not dst.exists():
        shutil.copy2(p, dst)

local_files = sorted(LOCAL_DIR.rglob("*"))
print("Local files copied:", len(local_files))

Local files copied: 300


In [35]:
from collections import Counter
suffixes = [p.suffix.lower() for p in audio_files]
print(Counter(suffixes).most_common(10))

[('.mid', 7920)]


Convert midi to wav file

In [36]:
!apt-get -qq update
!apt-get -qq install -y fluidsynth fluid-soundfont-gm
!pip -q install midi2audio librosa

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libdouble-conversion3:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../00-libdouble-conversion3_3.1.7-4_amd64.deb ...
Unpacking libdouble-conversion3:amd64 (3.1.7-4) ...
Selecting previously unselected package libqt5core5a:amd64.
Preparing to unpack .../01-libqt5core5a_5.15.3+dfsg-2ubuntu0.2_amd64.deb ...
Unpacking libqt5core5a:amd64 (5.15.3+dfsg-2ubuntu0.2) ...
Selecting previously unselected package libevdev2:amd64.
Preparing to unpack .../02-libevdev2_1.12.1+dfsg-1_amd64.deb ...
Unpacking libevdev2:amd64 (1.12.1+dfsg-1) ...
Selecting previously unselected package libmtdev1:amd64.
Preparing to unpack .../03-libmtdev1_1.1.6-1build4_amd64.deb ...
Unpacking libmtdev1:

In [37]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

MIDI_DIR = Path("/content/drive/MyDrive/1001-2000-midis")
assert MIDI_DIR.exists(), f"Not found: {MIDI_DIR}"

midi_files = sorted(MIDI_DIR.rglob("*.mid")) + sorted(MIDI_DIR.rglob("*.midi"))
print("MIDI files found:", len(midi_files))
print("Example:", midi_files[0] if midi_files else "None")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MIDI files found: 7920
Example: /content/drive/MyDrive/1001-2000-midis/1001_Balalaika.mid


In [39]:
import time
from pathlib import Path
from midi2audio import FluidSynth

# soundfont path used in the lab-friendly setup
SOUNDFONT = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
fs = FluidSynth(sound_font=SOUNDFONT, sample_rate=44100)

test_subset = midi_files[:5]

start = time.time()
tmp_dir = Path("/content/midi_wavs_test")
tmp_dir.mkdir(parents=True, exist_ok=True)

for p in test_subset:
    out_wav = tmp_dir / f"{p.stem}.wav"
    fs.midi_to_audio(str(p), str(out_wav))

elapsed = time.time() - start
print(f"5 files took: {elapsed:.1f}s (~{elapsed/5:.1f}s per file)")
print(f"Estimated 200 files: {(elapsed/5)*200/60:.1f} minutes")

5 files took: 7.1s (~1.4s per file)
Estimated 200 files: 4.7 minutes


In [ ]:
import random
from pathlib import Path
from midi2audio import FluidSynth

MAX_FILES = 200
subset = midi_files[:MAX_FILES]

LOCAL_WAV_DIR = Path("/content/midi_wavs")
LOCAL_WAV_DIR.mkdir(parents=True, exist_ok=True)

SOUNDFONT = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
fs = FluidSynth(sound_font=SOUNDFONT, sample_rate=44100)

wav_paths = []
for p in subset:
    out_wav = LOCAL_WAV_DIR / f"{p.stem}.wav"
    if not out_wav.exists():
        fs.midi_to_audio(str(p), str(out_wav))
    wav_paths.append(out_wav)

print("WAVs ready:", len(wav_paths))
print("Example:", wav_paths[0])

import time

start = time.time()
for i, p in enumerate(subset, start=1):
    out_wav = LOCAL_WAV_DIR / f"{p.stem}.wav"
    if not out_wav.exists():
        fs.midi_to_audio(str(p), str(out_wav))

    if i % 10 == 0:
        elapsed = time.time() - start
        print(f"{i}/{len(subset)} done | elapsed {elapsed/60:.1f} min | avg {elapsed/i:.2f}s/file")

Results:
 WAVs ready: 200
Example: /content/midi_wavs/1001_Balalaika.wav
10/200 done | elapsed 0.0 min | avg 0.00s/file
20/200 done | elapsed 0.0 min | avg 0.00s/file
30/200 done | elapsed 0.0 min | avg 0.00s/file
40/200 done | elapsed 0.0 min | avg 0.00s/file
50/200 done | elapsed 0.0 min | avg 0.00s/file
60/200 done | elapsed 0.0 min | avg 0.00s/file
70/200 done | elapsed 0.0 min | avg 0.00s/file
80/200 done | elapsed 0.0 min | avg 0.00s/file
90/200 done | elapsed 0.0 min | avg 0.00s/file
100/200 done | elapsed 0.0 min | avg 0.00s/file
110/200 done | elapsed 0.0 min | avg 0.00s/file
120/200 done | elapsed 0.0 min | avg 0.00s/file
130/200 done | elapsed 0.0 min | avg 0.00s/file
140/200 done | elapsed 0.0 min | avg 0.00s/file
150/200 done | elapsed 0.0 min | avg 0.00s/file
160/200 done | elapsed 0.0 min | avg 0.00s/file
170/200 done | elapsed 0.0 min | avg 0.00s/file
180/200 done | elapsed 0.0 min | avg 0.00s/file
190/200 done | elapsed 0.0 min | avg 0.00s/file
200/200 done | elapsed 0.0 min | avg 0.00s/file

In [41]:
import os
import librosa

print("Example WAV size (bytes):", os.path.getsize(wav_paths[0]))

y, sr = librosa.load(wav_paths[0], sr=44100, mono=True)
print("Loaded audio:", y.shape, "sr:", sr, "duration(s):", len(y) / sr)

Example WAV size (bytes): 22365996
Loaded audio: (5591488,) sr: 44100 duration(s): 126.7911111111111


In [44]:
from pathlib import Path
import numpy as np
import librosa

SR = 44100
SEGMENT_SECONDS = 10
SEGMENT_SAMPLES = SR * SEGMENT_SECONDS

def load_audio_fixed(path: str | Path, sr: int = SR, n_samples: int = SEGMENT_SAMPLES) -> np.ndarray:
    """Load first N samples (pads with zeros if needed) to keep batch shapes consistent."""
    y, _ = librosa.load(path, sr=sr, mono=True, duration=n_samples / sr)
    if len(y) < n_samples:
        y = np.pad(y, (0, n_samples - len(y)))
    else:
        y = y[:n_samples]
    return y.astype(np.float32)

In [45]:
from pathlib import Path
import numpy as np

BATCH_SIZE = 16

ids = []
embeddings = []

for start in range(0, len(wav_paths), BATCH_SIZE):
    batch_paths = wav_paths[start:start + BATCH_SIZE]
    batch_audio = np.stack([load_audio_fixed(p) for p in batch_paths], axis=0)  # (B, samples)

    # PANNs inference: returns (clipwise_output, embedding)
    _, batch_emb = model.inference(batch_audio)  # expected (B, 2048)

    embeddings.append(batch_emb)
    ids.extend([Path(p).stem for p in batch_paths])

embeddings = np.vstack(embeddings)
print("Embeddings shape:", embeddings.shape)
print("First id:", ids[0])

Embeddings shape: (200, 2048)
First id: 1001_Balalaika


In [46]:
to_upsert = []
for i, vec in enumerate(embeddings):
    meta = {
        "source": "midi_render",
        "filename": f"{ids[i]}.wav",
        "segment_seconds": SEGMENT_SECONDS,
        "sr": SR,
    }
    to_upsert.append((ids[i], vec.tolist(), meta))

index.upsert(vectors=to_upsert, namespace="midi")
print("Upserted:", len(to_upsert))
print(index.describe_index_stats())

Upserted: 200
{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 2000}, 'midi': {'vector_count': 200}},
 'total_vector_count': 2200}


In [ ]:
import random
from IPython.display import Audio, display

def recommend_from_wav(query_path, top_k=5):
    q_audio = load_audio_fixed(query_path)
    _, q_emb = model.inference(q_audio[np.newaxis, :])  # (1, 2048)
    res = index.query(
        vector=q_emb[0].tolist(),
        top_k=top_k,
        namespace="midi",
        include_metadata=True,
    )
    return res

# pick a random track as the query
query_path = random.choice(wav_paths)
print("Query:", Path(query_path).name)
display(Audio(str(query_path)))

results = recommend_from_wav(query_path, top_k=5)

for match in results["matches"]:
    mid = match["id"]
    score = match["score"]
    wav_file = Path("/content/midi_wavs") / f"{mid}.wav"
    print(f"- {mid} | score={score:.4f} | exists={wav_file.exists()}")
    if wav_file.exists():
        display(Audio(str(wav_file)))

Result from Cell above (mitigate git hub file size restriction)

Query: 1004_Drums.wav
- 1004_Drums | score=1.0000 | exists=True
- 1006_Drums | score=0.9893 | exists=True
- 1019_Drums | score=0.9596 | exists=True
- 1014_Drums | score=0.9573 | exists=True
- 1017_Drums | score=0.9556 | exists=True

Biuld Reconmendation Model

In [48]:
from pathlib import Path

WAV_DIR = Path("/content/midi_wavs")
id_to_path = {p.stem: p for p in WAV_DIR.glob("*.wav")}

print("WAV_DIR:", WAV_DIR)
print("Files indexed:", len(id_to_path))
print("Example:", next(iter(id_to_path.items())) if id_to_path else "None")

WAV_DIR: /content/midi_wavs
Files indexed: 200
Example: ('1014_Drums', PosixPath('/content/midi_wavs/1014_Drums.wav'))


In [49]:
def format_candidates(matches, max_candidates=8) -> str:
    lines = []
    for m in matches[:max_candidates]:
        clip_id = m["id"]
        score = float(m["score"])
        meta = m.get("metadata", {}) or {}
        filename = meta.get("filename", f"{clip_id}.wav")
        lines.append(f"- clip_id: {clip_id} | score: {score:.4f} | filename: {filename}")
    return "\n".join(lines)

In [51]:
from transformers import pipeline

# Small, fast text model (works in Colab). If you already have one, keep it.
text_gen = pipeline(
    "text-generation",
    model="google/flan-t5-base",   # instruction-following
    tokenizer="google/flan-t5-base",
    max_new_tokens=220,
)

PROMPT_TEMPLATE = """You are a music/sound recommendation assistant.

Goal:
Given a user query and a shortlist of candidate audio clips (with similarity scores and metadata),
pick the best 3 sound bytes to recommend.

Rules:
- Do not invent clips. Only choose from the candidates provided.
- Explain choices briefly and reference the clip_id for each recommendation.
- Keep output short and actionable.

User query:
{user_query}

Candidates (each has clip_id, score, filename):
{candidates}

Return:
1) 3 recommended sound bytes with: clip_id, score, and a 1-sentence reason.
2) A 1-sentence summary of the overall vibe/fit.
"""

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

In [53]:
id="t5_setup"
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

T5_MODEL_NAME = "google/flan-t5-base"

t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL_NAME)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
t5_model = t5_model.to(device)

def t5_generate(prompt: str, max_new_tokens: int = 220) -> str:
    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = t5_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [58]:
import numpy as np

def embed_wav_for_query(wav_path: str) -> np.ndarray:
    q_audio = load_audio_fixed(wav_path)               # from earlier cell (10s fixed)
    _, q_emb = model.inference(q_audio[np.newaxis, :]) # (1, 2048)
    return q_emb[0]

def retrieve_candidates(wav_path: str, top_k: int = 8):
    q_vec = embed_wav_for_query(wav_path).tolist()
    res = index.query(
        vector=q_vec,
        top_k=top_k,
        namespace="midi",
        include_metadata=True,
    )
    return res["matches"]

def recommend_sound_bytes(user_query: str, wav_path: str, top_k: int = 8) -> str:
    matches = retrieve_candidates(wav_path, top_k=top_k)
    candidates_txt = format_candidates(matches, max_candidates=top_k)

    # Keep your Prompt 1, but add a strict-ish output format to make parsing easy
    prompt = PROMPT_TEMPLATE.format(user_query=user_query, candidates=candidates_txt) + """
Output format (must follow):
1) clip_id: <ID> | score: <SCORE> | reason: <ONE SENTENCE>
2) clip_id: <ID> | score: <SCORE> | reason: <ONE SENTENCE>
3) clip_id: <ID> | score: <SCORE> | reason: <ONE SENTENCE>
Summary: <ONE SENTENCE>
"""

    return t5_generate(prompt, max_new_tokens=220)

In [59]:
from pathlib import Path
import librosa
import numpy as np
from IPython.display import Audio, display

WAV_DIR = Path("/content/midi_wavs")

def load_audio_segment(path: str | Path, sr: int = 44100, start_sec: float = 10.0, duration_sec: float = 10.0) -> np.ndarray:
    y, _ = librosa.load(path, sr=sr, mono=True, offset=float(start_sec), duration=float(duration_sec))
    return y.astype(np.float32)

def _safe_duration_sec(path: str | Path) -> float:
    # librosa API differs across versions; keep it robust
    try:
        return float(librosa.get_duration(path=str(path)))
    except TypeError:
        return float(librosa.get_duration(filename=str(path)))

def pick_recommended_ids(response_text: str, matches, n: int = 3, exclude_id: str | None = None):
    candidate_ids = [m["id"] for m in matches]
    found = []
    for cid in candidate_ids:
        if exclude_id and cid == exclude_id:
            continue
        pos = response_text.find(cid)
        if pos != -1:
            found.append((pos, cid))

    found_ids = [cid for _, cid in sorted(found, key=lambda x: x[0])]

    # fallback: if model didn’t include ids cleanly, use top matches
    if len(found_ids) < n:
        for m in matches:
            cid = m["id"]
            if exclude_id and cid == exclude_id:
                continue
            if cid not in found_ids:
                found_ids.append(cid)
            if len(found_ids) >= n:
                break

    return found_ids[:n]

def recommend_and_play(user_query: str, query_wav_path: str, top_k: int = 8, byte_seconds: float = 10.0, start_sec_default: float = 10.0):
    query_id = Path(query_wav_path).stem

    # 1) retrieve
    matches = retrieve_candidates(query_wav_path, top_k=top_k)

    # 2) LLM chooses
    response = recommend_sound_bytes(user_query, query_wav_path, top_k=top_k)

    print("\n" + "=" * 90)
    print("USER QUERY:", user_query)
    print("QUERY AUDIO:", Path(query_wav_path).name)
    print("\nASSISTANT RESPONSE:\n")
    print(response)

    # 3) play query sound byte
    print("\nQuery sound byte:")
    display(Audio(load_audio_segment(query_wav_path, start_sec=start_sec_default, duration_sec=byte_seconds), rate=44100))

    # 4) parse clip_ids and play recommended bytes
    rec_ids = pick_recommended_ids(response, matches, n=3, exclude_id=query_id)

    print("\nRecommended sound bytes:")
    for cid in rec_ids:
        wav_file = WAV_DIR / f"{cid}.wav"
        exists = wav_file.exists()
        print(f"- {cid} | exists={exists}")
        if not exists:
            continue

        dur = _safe_duration_sec(wav_file)
        # pick a safe start (avoid going past end)
        start_sec = min(start_sec_default, max(0.0, dur - byte_seconds))
        y = load_audio_segment(wav_file, start_sec=start_sec, duration_sec=byte_seconds)
        display(Audio(y, rate=44100))

In [ ]:
import random

query_path = random.choice(wav_paths)

tests = [
    "Recommend the 3 most similar sound bytes to this query.",
    "Pick 3: one closest match, one medium match, one slightly different but still relevant.",
    "I want something calm and background-friendly; select 3 suitable sound bytes.",
]

for q in tests:
    recommend_and_play(q, str(query_path), top_k=8, byte_seconds=10.0, start_sec_default=10.0)

Results from above cell (mitigate git hub file size restriction): ==========================================================================================
USER QUERY: Recommend the 3 most similar sound bytes to this query.
QUERY AUDIO: 1025_AcousticGuitar.wav

ASSISTANT RESPONSE:

clip_id: ID> | score: SCORE> | reason: ONE SENTENCE>

Query sound byte:

Recommended sound bytes:
- 1010_Balalaika | exists=True
- 1001_Ukulele | exists=True
- 1011_Balalaika | exists=True

==========================================================================================
USER QUERY: Pick 3: one closest match, one medium match, one slightly different but still relevant.
QUERY AUDIO: 1025_AcousticGuitar.wav

ASSISTANT RESPONSE:

clip_id: ID> | score: SCORE> | reason: ONE SENTENCE>

Query sound byte:

Recommended sound bytes:
- 1010_Balalaika | exists=True
- 1001_Ukulele | exists=True
- 1011_Balalaika | exists=True

==========================================================================================
USER QUERY: I want something calm and background-friendly; select 3 suitable sound bytes.
QUERY AUDIO: 1025_AcousticGuitar.wav

ASSISTANT RESPONSE:

clip_id: ID> | score: SCORE> | reason: ONE SENTENCE>

Query sound byte:

Recommended sound bytes:
- 1010_Balalaika | exists=True
- 1001_Ukulele | exists=True
- 1011_Balalaika | exists=True

## Lab Summary — Music Recommendation Assistant (Audio Similarity + LLM Reranking)

### What I built
In this lab I built a simple **music/sound recommendation assistant** using an **audio embedding model + vector search**, then added a small **LLM “presentation layer”** that selects and explains the best matches and plays short **sound bytes**.

Pipeline (end-to-end):
1. **Dataset ingestion**
   - I used a local dataset of **MIDI files** (4598 `.mid` files) stored in Google Drive.
2. **MIDI → Audio conversion**
   - Because the embedding model expects **audio waveforms**, I converted a **subset** of MIDIs to `.wav` using **FluidSynth** + a GM SoundFont.
3. **Audio embeddings**
   - I generated 2048-dimensional embeddings using **PANNs** via `panns_inference.AudioTagging`.
   - To keep runtime reasonable, I embedded a fixed-length segment (e.g., first 10 seconds) per track.
4. **Vector indexing + retrieval**
   - I stored embeddings in **Pinecone** (cosine similarity) under a dedicated namespace (e.g., `midi`) to avoid mixing with other data.
   - Given a query audio file, I embedded it and retrieved **top-k** nearest neighbors with `index.query(...)`.
5. **Recommendation assistant output**
   - I used a lightweight instruction model (**FLAN-T5**) to:
     - choose the best **3** recommendations from the retrieved candidates,
     - provide brief reasons and a short summary.
   - The notebook then displayed **inline playable sound bytes** (10-second clips) for the query and the selected recommendations.

---

## Key Takeaways

### 1) Audio embeddings enable “semantic” similarity search
- Instead of comparing filenames or metadata, we compare **embedding vectors** derived from the audio signal.
- This makes it possible to retrieve similar sounds even when the dataset has minimal labels.

### 2) Data format matters (MIDI vs audio)
- The PANNs model works on **audio**, not symbolic MIDI.
- To use MIDI in this pipeline, I needed a conversion step (MIDI → WAV).  
  This also highlights that real systems must handle **format compatibility** early.

### 3) Vector DB workflow: embed → upsert → query
- Pinecone usage followed the standard pattern:
  - create/connect index,
  - upsert vectors (id + embedding + metadata),
  - query with a new embedding and return the nearest neighbors.

### 4) LLMs are best used as a “selector + explainer” here
- The LLM **cannot hear audio** and does not generate sound.
- Its value is in:
  - **choosing** among retrieved candidates (especially when you want diversity),
  - producing **human-friendly explanations** and a clean output format for a UI.

### 5) Colab reliability: environment + credentials are common failure points
- Colab runtime restarts reset environment variables, so API keys must be re-loaded (Drive secrets / `getpass()`).
- Avoid silent fallbacks like `"YOUR_API_KEY"` because they hide the real issue (authentication).

---

## What I would improve next (optional)
- Store richer metadata during upsert (e.g., duration, energy/RMS, tempo estimate) to improve LLM explanations.
- Use a smarter “sound byte” selector (middle-of-track, random-but-deterministic, or energy-based segment selection).
- Scale beyond a subset of files by batching conversions/embeddings and using more efficient storage/processing.

---

## Final Result
A working prototype that:
- converts MIDI → WAV (subset),
- embeds audio with PANNs,
- stores/retrieves vectors in Pinecone,
- produces a readable recommendation response,
- and plays short audio snippets inline for quick evaluation.